# Проект: предсказания победителя в онлайн-игре

## Первый этап

In [18]:
import time

import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_rows", None)  # or a specific number
pd.set_option("display.max_columns", None)  # to show all columns
pd.set_option("display.expand_frame_repr", False)  # to allow wider DataFrame display

In [19]:
features = pd.read_csv("features/features.csv", index_col='match_id')

In [20]:
nafeat = features.isna().any()
print(*list(nafeat[nafeat].index), sep=", ")

first_blood_time, first_blood_team, first_blood_player1, first_blood_player2, radiant_bottle_time, radiant_courier_time, radiant_flying_courier_time, radiant_first_ward_time, dire_bottle_time, dire_courier_time, dire_flying_courier_time, dire_first_ward_time


In [21]:
target_col = "radiant_win"
features_to_remove = [
    "duration",
    "tower_status_radiant",
    "tower_status_dire",
    "barracks_status_dire",
    "barracks_status_radiant",
    "start_time",
    "lobby_type",
]
y = features[target_col].copy()
X = features.drop(features_to_remove + [target_col], axis=1)
X = X.fillna(0)

In [22]:
scaler = StandardScaler()

In [23]:
def run_gradient_boosting(X, y, n_estimators=30):
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    score, elapsed_time = [], []
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        gbc = GradientBoostingClassifier(n_estimators=n_estimators, random_state=241)
        gbc.fit(X[train_ind], y[train_ind])
        y_pred_proba = gbc.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        elapsed_time.append(int(time.time() - start))
        score.append(auc_roc)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).sum()

In [24]:
for n_estimators in [10, 20, 30, 50]:
    score_mean, score_std, elapsed_time_total = run_gradient_boosting(scaler.fit_transform(X), y.to_numpy(), n_estimators)
    print(f"n_estimators={n_estimators} | AUC-ROC (mean, std): {score_mean}, {score_std} | mean time: {elapsed_time_total} sec")

n_estimators=10 | AUC-ROC (mean, std): 0.6643666526805019, 0.004881109172169747 | mean time: 40 sec
n_estimators=20 | AUC-ROC (mean, std): 0.6828403189744824, 0.004994965985175589 | mean time: 84 sec
n_estimators=30 | AUC-ROC (mean, std): 0.6895522791393396, 0.004530518804238089 | mean time: 126 sec
n_estimators=50 | AUC-ROC (mean, std): 0.6974344690919826, 0.003977555235729339 | mean time: 212 sec


## отчет по этапу

1. Какие признаки имеют пропуски среди своих значений? Что могут означать пропуски в этих признаках (ответьте на этот вопрос для двух любых признаков)?

    first_blood_time, first_blood_team, first_blood_player1, first_blood_player2, radiant_bottle_time, radiant_courier_time, radiant_flying_courier_time, radiant_first_ward_time, dire_bottle_time, dire_courier_time, dire_flying_courier_time, dire_first_ward_time

    first_blood_time == NaN можно встретить напрмер в партии с индексом 3. Ни один из игроков не совершил ни одного убийства поэтому ивента first_blood не произошло. Соответственно нет и команды которая совершила первое убийство в партии (first_blood_team).

2. Как называется столбец, содержащий целевую переменную?

    radiant_win

3. Как долго проводилась кросс-валидация для градиентного бустинга с 30 деревьями? Инструкцию по измерению времени можно найти ниже по тексту. Какое качество при этом получилось? Напомним, что в данном задании мы используем метрику качества AUC-ROC.

    Время для кросс-валидации с 30 деревьями составило ~= 130s с качеством AUC-ROC ~= 0.69

4. Имеет ли смысл использовать больше 30 деревьев в градиентном бустинге? Что бы вы предложили делать, чтобы ускорить его обучение при увеличении количества деревьев?
   
    При увеличение числа деревьев с 30 до 50 качество выростло с 0.69 до 0.70, минусом является увеличение так же времени обучения. Если время обучения остается удовлетворительным, то да, стоит. Для ускорения обучения при увеличении количества деревьев можно уменьшить их размер, параметром max_depth или использовать раннюю остановку мониторя валидационный лосс.

## Второй этап

In [ ]:
def run_logistic_regression(X, y, C=1.0):
    score, elapsed_time = [], []
    kf = KFold(n_splits=5, shuffle=True, random_state=241)
    for train_ind, test_ind in kf.split(X, y):
        start = time.time()
        lg = LogisticRegression(
            random_state=241,
            penalty="l2",
            C=C,
            max_iter=1000
        )
        lg.fit(X[train_ind], y[train_ind])
        y_pred_proba = lg.predict_proba(X[test_ind])[:, 1]
        auc_roc = roc_auc_score(y_true=y[test_ind], y_score=y_pred_proba)
        score.append(auc_roc)
        elapsed_time.append(time.time() - start)
    return np.array(score).mean(), np.array(score).std(), np.array(elapsed_time).sum()

In [26]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | total time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7161, 0.0027 | total time: 3.29 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.14 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.00 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 3.97 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.78 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.77 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.08 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7162, 0.0028 | total time: 4.62 sec


In [47]:
categorial_features = [
    "r1_hero",
    "r2_hero",
    "r3_hero",
    "r4_hero",
    "r5_hero",
    "d1_hero",
    "d2_hero",
    "d3_hero",
    "d4_hero",
    "d5_hero",
]

In [48]:
X_without_cat = X.drop(categorial_features, axis=1)

In [49]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X_without_cat), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | mean time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7161, 0.0027 | mean time: 3.19 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 3.67 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 5.32 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 4.67 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 4.85 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 4.51 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 5.78 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7163, 0.0028 | mean time: 5.14 sec


In [50]:
unique_heros = set()
for f in categorial_features:
    unique_heros |= set(features[f].unique())

In [51]:
print(f"Number of heroes in game: {max(unique_heros)}")
print(f"Number of unique heroes in dataset: {len(unique_heros)}")

Number of heroes in game: 112
Number of unique heroes in dataset: 108


In [31]:
X_pick = np.zeros((len(features), max(unique_heros)))
for i, match_id in enumerate(features.index):
    for p in range(1, 6):
        X_pick[i, int(features.at[match_id, f"r{p}_hero"])-1]  = 1
        X_pick[i, int(features.at[match_id, f"d{p}_hero"])-1]  = -1

In [32]:
X_pick = pd.DataFrame(X_pick, index=features.index, columns=[f"hero_{i+1}" for i in range(max(unique_heros))])

In [33]:
X_encoded_cat = pd.concat([X_without_cat, X_pick], axis=1)

In [34]:
for C in [10 ** i for i in range(-3, 5)]:
    score_mean, score_std, elapsed_time_total = run_logistic_regression(scaler.fit_transform(X_encoded_cat), y.to_numpy(), C)
    print(f"C={C:.1e} | AUC-ROC (mean, std): {score_mean:.4f}, {score_std:.4f} | mean time: {elapsed_time_total:.2f} sec")

C=1.0e-03 | AUC-ROC (mean, std): 0.7515, 0.0023 | mean time: 4.43 sec
C=1.0e-02 | AUC-ROC (mean, std): 0.7518, 0.0021 | mean time: 5.70 sec
C=1.0e-01 | AUC-ROC (mean, std): 0.7518, 0.0021 | mean time: 6.45 sec
C=1.0e+00 | AUC-ROC (mean, std): 0.7518, 0.0020 | mean time: 6.77 sec
C=1.0e+01 | AUC-ROC (mean, std): 0.7518, 0.0020 | mean time: 7.94 sec
C=1.0e+02 | AUC-ROC (mean, std): 0.7518, 0.0020 | mean time: 6.81 sec
C=1.0e+03 | AUC-ROC (mean, std): 0.7518, 0.0020 | mean time: 6.28 sec
C=1.0e+04 | AUC-ROC (mean, std): 0.7518, 0.0020 | mean time: 6.23 sec


## evaluate on test dataset

In [56]:
features_test = pd.read_csv("features_test/features_test.csv", index_col='match_id')

In [57]:
X_test = features_test.drop(["start_time", "lobby_type"], axis=1)
X_test = X_test.fillna(0)

In [58]:
X_test_pick = np.zeros((len(features_test), max(unique_heros)))
for i, match_id in enumerate(features_test.index):
    for p in range(1, 6):
        X_test_pick[i, int(features_test.at[match_id, f"r{p}_hero"]) - 1] = 1
        X_test_pick[i, int(features_test.at[match_id, f"d{p}_hero"]) - 1] = -1
X_test_pick = pd.DataFrame(
    X_test_pick,
    index=features_test.index,
    columns=[f"hero_{i + 1}" for i in range(max(unique_heros))],
)
X_test_encoded_cat = pd.concat(
    [X_test.drop(categorial_features, axis=1), X_test_pick], axis=1
)

In [59]:
lg = LogisticRegression(
    random_state=241,
    penalty="l2",
    C=1,
    max_iter=1000
)
lg.fit(scaler.fit_transform(X_encoded_cat), y.to_numpy())

LogisticRegression(C=1, max_iter=1000, random_state=241)

In [73]:
y_pred_test = lg.predict_proba(scaler.fit_transform(X_test_encoded_cat))[:, 1]

In [74]:
float(y_pred_test.min()), float(y_pred_test.max())

(0.008722367260230353, 0.9966511999796765)

## отчет по этапу

1. Какое качество получилось у логистической регрессии над всеми исходными признаками? Как оно соотносится с качеством градиентного бустинга? Чем вы можете объяснить эту разницу? Быстрее ли работает логистическая регрессия по сравнению с градиентным бустингом?

    метрки лог регрессии над всеми признаками получилась ~ 0.7162. что на 0.02 больше чем у градиентного бустинга с 50 деревьями. Разница относительно небольшая, видимо градиентный бустниг больше склонен к переобучению на данном размере датасета. Лог регрессия обучается значительно быстрее - 3с против 212с

2. Как влияет на качество логистической регрессии удаление категориальных признаков (укажите новое значение метрики качества)? Чем вы можете объяснить это изменение?

    после удаления категориальных признаков метрика почти не изменилась: стало 0.7163 было 0.7162. Признаков много, влияние удаленных было около нулевым

3. Сколько различных идентификаторов героев существует в данной игре?

    112 в игре, уникальных героев в датасете - 108

4. Какое получилось качество при добавлении "мешка слов" по героям? Улучшилось ли оно по сравнению с предыдущим вариантом? Чем вы можете это объяснить?

    после добавления "мешка слов" по героям - метрка значительно выросла 0.7515 против 0.7163. увеличение количества фич увеличила размерность пространства, где логистической регрессии удалось лучше разделить данные гиперповерхностью

5. Какое минимальное и максимальное значение прогноза на тестовой выборке получилось у лучшего из алгоритмов?

    мин 0.0087, макс 0.9966
